In [2]:
import pandas as pd
from scipy import stats

df = pd.read_csv('retail_sales_dataset.csv')


df_clean = df.dropna(subset=['Product Category', 'Total Amount'])
df_clean = df_clean[df_clean['Total Amount'] > 0]

print(f"Cleaned dataset shape: {df_clean.shape}")


categories = df_clean['Product Category'].unique()

print("\n--- Shapiro-Wilk Normality Test Results ---")
for category in categories:
    
    group_data = df_clean[df_clean['Product Category'] == category]['Total Amount']
    
   
    stat, p_value = stats.shapiro(group_data)
    
    print(f"{category}: Statistics={stat:.3f}, p-value={p_value:.3e}")
    if p_value > 0.05:
        print(f"  -> {category} looks normally distributed (Fail to reject H0)")
    else:
        print(f"  -> {category} does NOT look normally distributed (Reject H0)")

Cleaned dataset shape: (1000, 9)

--- Shapiro-Wilk Normality Test Results ---
Beauty: Statistics=0.758, p-value=6.339e-21
  -> Beauty does NOT look normally distributed (Reject H0)
Clothing: Statistics=0.742, p-value=5.329e-23
  -> Clothing does NOT look normally distributed (Reject H0)
Electronics: Statistics=0.745, p-value=1.386e-22
  -> Electronics does NOT look normally distributed (Reject H0)


In [3]:
beauty = df_clean[df_clean['Product Category'] == 'Beauty']['Total Amount']
clothing = df_clean[df_clean['Product Category'] == 'Clothing']['Total Amount']
electronics = df_clean[df_clean['Product Category'] == 'Electronics']['Total Amount']


stat_lev, p_lev = stats.levene(beauty, clothing, electronics)
print(f"Levene's Test for Equal Variances: p-value = {p_lev:.3f}")
if p_lev < 0.05:
    print("  -> Variances are significantly different (Violates ANOVA assumption)\n")
else:
    print("  -> Variances are equal (Passes assumption)\n")


f_stat, anova_p = stats.f_oneway(beauty, clothing, electronics)
print(f"One-Way ANOVA Results: F-statistic = {f_stat:.3f}, p-value = {anova_p:.3f}")


h_stat, kruskal_p = stats.kruskal(beauty, clothing, electronics)
print(f"Kruskal-Wallis Results: H-statistic = {h_stat:.3f}, p-value = {kruskal_p:.3f}")

print("\n--- FINAL CONCLUSION ---")
if kruskal_p < 0.05:
    print("We REJECT the null hypothesis.")
    print("There IS a statistically significant difference in the average Total Amount spent across categories.")
else:
    print("We FAIL TO REJECT the null hypothesis.")
    print("There is NO statistically significant difference in the average Total Amount spent across categories.")

Levene's Test for Equal Variances: p-value = 0.814
  -> Variances are equal (Passes assumption)

One-Way ANOVA Results: F-statistic = 0.159, p-value = 0.853
Kruskal-Wallis Results: H-statistic = 0.055, p-value = 0.973

--- FINAL CONCLUSION ---
We FAIL TO REJECT the null hypothesis.
There is NO statistically significant difference in the average Total Amount spent across categories.


In [ ]:
## Final Conclusion
Based on the analysis, the Shapiro-Wilk test indicated that the data violates the assumption of normality. However, Levene's test confirmed equal variances across groups (p = 0.814). 

Because of the non-normality, we rely on the non-parametric Kruskal-Wallis H test (p = 0.973) alongside the parametric One-Way ANOVA (p = 0.853). Both tests yielded a p-value strictly greater than our alpha level of 0.05. 

**Verdict:** We fail to reject the null hypothesis. There is no statistically significant difference in the average transaction amount across the Beauty, Clothing, and Electronics categories.